# SFT Training Pairs — EDA & Validation

Explores and validates the format of `sft_pairs.parquet` before using it for SFT and GRPO training.

Key checks:
1. Schema: messages format, required fields, types
2. Completions: JSON validity, CI tuple completeness, reasoning quality
3. Distribution: per-book balance, flow counts, token lengths
4. Edge cases: empty fields, malformed JSON, missing components

In [1]:
import pandas as pd
import json
import numpy as np
from collections import Counter
from IPython.display import display

SFT_PATH = '/share/pierson/matt/UAIR/multirun/2026-03-23_grpo_training/22-11-45/sft_only/outputs/sft_data/sft_pairs.parquet'
df = pd.read_parquet(SFT_PATH)
print(f'Total SFT pairs: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'dtypes:\n{df.dtypes}')

Total SFT pairs: 772
Columns: ['messages', 'source_id', 'task_type']
dtypes:
messages     object
source_id    object
task_type    object
dtype: object


## 1. Schema Validation

In [2]:
# Parse all messages and validate structure
parse_errors = []
parsed = []

for idx, row in df.iterrows():
    try:
        msgs = json.loads(row['messages'])
        assert isinstance(msgs, list), 'messages is not a list'
        assert len(msgs) == 2, f'expected 2 messages, got {len(msgs)}'
        assert msgs[0]['role'] == 'user', f'first message role is {msgs[0]["role"]}'
        assert msgs[1]['role'] == 'assistant', f'second message role is {msgs[1]["role"]}'
        
        user_content = msgs[0]['content']
        assistant_content = msgs[1]['content']
        completion = json.loads(assistant_content)
        
        parsed.append({
            'idx': idx,
            'source_id': row['source_id'],
            'user_len': len(user_content),
            'assistant_len': len(assistant_content),
            'has_reasoning': bool(completion.get('reasoning')),
            'has_exchange': completion.get('has_information_exchange'),
            'n_flows': len(completion.get('flows', [])),
            'completion': completion,
        })
    except Exception as e:
        parse_errors.append({'idx': idx, 'error': str(e)})

pdf = pd.DataFrame(parsed)
print(f'Parsed: {len(pdf):,} / {len(df):,} ({len(parse_errors)} errors)')
if parse_errors:
    print(f'\nParse errors:')
    for e in parse_errors[:5]:
        print(f'  Row {e["idx"]}: {e["error"]}')

Parsed: 772 / 772 (0 errors)


## 2. Per-Book Distribution

In [3]:
ID_TO_TITLE = {
    '1984': '1984', '541': 'Age of Innocence', '11': 'Alice',
    '1399': 'Anna Karenina', '1023': 'Bleak House',
    '1184': 'Count of Monte Cristo', '135': 'Les Mis',
    '145': 'Middlemarch', '4078': 'Dorian Gray', '1342': 'Pride & Prejudice',
}

book_stats = pdf.groupby('source_id').agg(
    count=('idx', 'count'),
    mean_flows=('n_flows', 'mean'),
    total_flows=('n_flows', 'sum'),
    mean_user_len=('user_len', 'mean'),
    mean_assistant_len=('assistant_len', 'mean'),
    pct_has_exchange=('has_exchange', 'mean'),
).round(2)
book_stats.index = book_stats.index.map(lambda x: ID_TO_TITLE.get(x, x))
display(book_stats.sort_values('count', ascending=False))

print(f'\nTotal flows across all pairs: {pdf["n_flows"].sum():,}')
print(f'Pairs with 0 flows: {(pdf["n_flows"] == 0).sum()}')
print(f'Pairs with has_information_exchange=False: {(pdf["has_exchange"] == False).sum()}')

,count,mean_flows,total_flows,mean_user_len,mean_assistant_len,pct_has_exchange
source_id,,,,,,
Les Mis,170,1.50,255,6098.72,1855.41,0.53
Count of Monte Cristo,167,2.14,358,6243.83,2393.69,0.59
Bleak House,110,1.91,210,6174.51,2249.59,0.53
Middlemarch,102,1.25,128,5979.56,1596.12,0.49
Anna Karenina,84,0.69,58,6084.90,1091.57,0.27
1984,51,2.75,140,5935.33,3236.94,0.71
Pride & Prejudice,41,1.20,49,6006.68,1564.56,0.41
Age of Innocence,27,0.74,20,6085.89,1068.41,0.30
Dorian Gray,15,1.13,17,5893.73,1485.00,0.33



Total flows across all pairs: 1,241
Pairs with 0 flows: 386
Pairs with has_information_exchange=False: 386


## 3. Flow Tuple Completeness
Check that all CI 5-tuple components are present and non-empty.

In [4]:
TUPLE_FIELDS = ['sender', 'recipient', 'subject', 'information_type', 'transmission_principle']
META_FIELDS = ['context', 'appropriateness', 'norms_invoked', 'norm_source', 'is_new_flow', 'confidence']
ALL_FIELDS = TUPLE_FIELDS + META_FIELDS

field_present = {f: 0 for f in ALL_FIELDS}
field_nonempty = {f: 0 for f in ALL_FIELDS}
total_flows = 0

for _, row in pdf.iterrows():
    for flow in row['completion'].get('flows', []):
        total_flows += 1
        for f in ALL_FIELDS:
            val = flow.get(f)
            if val is not None:
                field_present[f] += 1
                if isinstance(val, str) and val.strip():
                    field_nonempty[f] += 1
                elif isinstance(val, (list, bool, int, float)):
                    field_nonempty[f] += 1

print(f'Total flows: {total_flows:,}')
print(f'\n{"Field":30s} {"Present":>8s} {"Non-empty":>10s} {"% Complete":>10s}')
print('-' * 60)
for f in ALL_FIELDS:
    pct = field_nonempty[f] / total_flows * 100 if total_flows > 0 else 0
    marker = '  ⚠' if pct < 95 else ''
    print(f'{f:30s} {field_present[f]:8d} {field_nonempty[f]:10d} {pct:9.1f}%{marker}')

Total flows: 1,241

Field                           Present  Non-empty % Complete
------------------------------------------------------------
sender                             1241       1241     100.0%
recipient                          1241       1241     100.0%
subject                            1178       1178      94.9%  ⚠
information_type                   1241       1241     100.0%
transmission_principle             1241       1241     100.0%
context                            1241       1241     100.0%
appropriateness                    1241       1241     100.0%
norms_invoked                      1241       1241     100.0%
norm_source                        1241       1241     100.0%
is_new_flow                        1241       1241     100.0%
confidence                         1241       1241     100.0%


## 4. Reasoning Quality

In [5]:
reasoning_lens = []
for _, row in pdf.iterrows():
    r = row['completion'].get('reasoning', '')
    reasoning_lens.append(len(r))

rl = pd.Series(reasoning_lens)
print('Reasoning length (chars):')
print(rl.describe().round(0))
print(f'\nEmpty reasoning: {(rl == 0).sum()}')
print(f'Short reasoning (<50 chars): {(rl < 50).sum()}')

# Show a few short ones
short_idx = rl[rl < 50].index[:3]
if len(short_idx) > 0:
    print(f'\nShort reasoning samples:')
    for i in short_idx:
        print(f'  [{pdf.iloc[i]["source_id"]}] "{pdf.iloc[i]["completion"]["reasoning"][:100]}"')

Reasoning length (chars):
count     772.0
mean     1105.0
std      1012.0
min       261.0
25%       400.0
50%       520.0
75%      1660.0
max      5774.0
dtype: float64

Empty reasoning: 0
Short reasoning (<50 chars): 0


## 5. Token Length Estimation
Estimate prompt+completion token counts to check for context window issues during training.

In [6]:
# Rough token estimate: chars / 4
pdf['est_user_tokens'] = pdf['user_len'] // 4
pdf['est_assistant_tokens'] = pdf['assistant_len'] // 4
pdf['est_total_tokens'] = pdf['est_user_tokens'] + pdf['est_assistant_tokens']

print('Estimated token counts:')
print(pdf[['est_user_tokens', 'est_assistant_tokens', 'est_total_tokens']].describe().round(0))

# Check how many exceed typical training max_seq_length
for limit in [2048, 4096, 8192]:
    n_over = (pdf['est_total_tokens'] > limit).sum()
    print(f'  Pairs exceeding {limit} tokens: {n_over} ({n_over/len(pdf)*100:.1f}%)')

Estimated token counts:
       est_user_tokens  est_assistant_tokens  est_total_tokens
count            772.0                 772.0             772.0
mean            1526.0                 487.0            2013.0
std              120.0                 504.0             528.0
min              724.0                  81.0             963.0
25%             1506.0                 117.0            1672.0
50%             1567.0                 192.0            1739.0
75%             1596.0                 786.0            2324.0
max             1680.0                2803.0            4333.0
  Pairs exceeding 2048 tokens: 267 (34.6%)
  Pairs exceeding 4096 tokens: 2 (0.3%)
  Pairs exceeding 8192 tokens: 0 (0.0%)


## 6. Appropriateness & Norm Distribution

In [7]:
appr_counts = Counter()
norm_source_counts = Counter()
norms_per_flow = []
new_flow_count = 0

for _, row in pdf.iterrows():
    for flow in row['completion'].get('flows', []):
        appr_counts[flow.get('appropriateness', 'missing')] += 1
        norm_source_counts[flow.get('norm_source', 'missing')] += 1
        invoked = flow.get('norms_invoked', [])
        norms_per_flow.append(len(invoked) if isinstance(invoked, list) else 0)
        if flow.get('is_new_flow'):
            new_flow_count += 1

print('Appropriateness distribution:')
for k, v in appr_counts.most_common():
    print(f'  {k:15s}: {v:5d} ({v/sum(appr_counts.values())*100:.1f}%)')

print(f'\nNorm source distribution:')
for k, v in norm_source_counts.most_common():
    print(f'  {k:15s}: {v:5d}')

print(f'\nNorms invoked per flow: mean={np.mean(norms_per_flow):.1f}, max={max(norms_per_flow)}')
print(f'Flows with 0 norms invoked: {sum(1 for n in norms_per_flow if n == 0)}')
print(f'New flows (is_new_flow=True): {new_flow_count}')

Appropriateness distribution:
  appropriate    :   967 (77.9%)
  inappropriate  :   220 (17.7%)
  ambiguous      :    54 (4.4%)

Norm source distribution:
  explicit       :  1092
  implicit       :   146
  both           :     3

Norms invoked per flow: mean=1.0, max=3
Flows with 0 norms invoked: 1
New flows (is_new_flow=True): 5


## 7. TRL Compatibility Check
Verify the data works with TRL's expected format.

In [8]:
# TRL SFTTrainer expects a 'messages' column with list-of-dicts
# Our format stores messages as JSON strings — check they round-trip
sample = df.iloc[0]
msgs = json.loads(sample['messages'])

print('TRL compatibility checks:')
print(f'  messages is JSON string: {isinstance(sample["messages"], str)}')
print(f'  Parses to list: {isinstance(msgs, list)}')
print(f'  Has user + assistant roles: {[m["role"] for m in msgs]}')
print(f'  Content is string: {all(isinstance(m["content"], str) for m in msgs)}')

# Check that all rows parse
all_parse = all(
    isinstance(json.loads(row['messages']), list)
    for _, row in df.iterrows()
)
print(f'  All rows parse as JSON lists: {all_parse}')

# Check for the GRPO dataset builder
import sys 
sys.path.append('/share/pierson/matt/UAIR/dagspaces')
from grpo_training.stages.grpo_training import _build_grpo_dataset
dataset = _build_grpo_dataset(df.head(10),)
print(f'\nGRPO dataset builder test:')
print(f'  Rows: {len(dataset)}')
print(f'  Columns: {dataset.column_names}')
print(f'  Sample prompt length: {len(dataset[0]["prompt"])} chars')

TRL compatibility checks:
  messages is JSON string: True
  Parses to list: True
  Has user + assistant roles: ['user', 'assistant']
  Content is string: True
  All rows parse as JSON lists: True


TypeError: _build_grpo_dataset() missing 2 required positional arguments: 'tokenizer' and 'prompt_template'

## 8. Sample Inspection

In [9]:
# Show 2 complete examples from different books
for gid in ['1984', '1342']:
    row = pdf[pdf['source_id'] == gid].iloc[0]
    msgs = json.loads(df.iloc[row['idx']]['messages'])
    completion = row['completion']
    
    print(f'\n{"="*80}')
    print(f'  Source: {ID_TO_TITLE.get(gid, gid)}')
    print(f'{"="*80}')
    print(f'\nUSER PROMPT ({len(msgs[0]["content"])} chars):')
    print(msgs[0]['content'] + '...')
    print(f'\nREASONING:')
    print(completion['reasoning'] + '...')
    print(f'\nFLOWS ({len(completion["flows"])}):' )
    for i, flow in enumerate(completion['flows'][:3]):
        print(f'  Flow {i+1}:')
        print(f'    {flow["sender"]} → {flow["recipient"]}: {flow["information_type"]}')
        print(f'    Transmission: {flow["transmission_principle"]}')
        print(f'    Context: {flow["context"]} | Appropriateness: {flow["appropriateness"]}')
        print(f'    Norms: {flow["norms_invoked"]}')


  Source: 1984

USER PROMPT (5483 chars):
Analyze the following text passage for information flows using the Contextual Integrity framework. First, reason about what information exchanges are described — identify senders, recipients, subjects, information types, and transmission principles. Then provide a structured extraction of each information flow as a flat JSON object with the 5-component CI tuple (sender, recipient, subject, information_type, transmission_principle) and contextual metadata.

Project Gutenberg Australia

Title: Nineteen eighty-four
Author: George Orwell (pseudonym of Eric Blair) (1903-1950)
* A Project Gutenberg of Australia eBook *
eBook No.:  0100021.txt
Language:   English
Date first posted: August 2001
Date most recently updated: November 2008

Project Gutenberg of Australia eBooks are created from printed editions
which are in the public domain in Australia, unless a copyright notice
is included. We do NOT keep any eBooks in compliance with a particular
pape